<img src="https://raw.githubusercontent.com/Databricks-BR/lab_agosto_2025/main/images/head_lab.png">

## LAB 02 — Entendendo os dados com o Assistant (CRM / lead scorer)

1. Rode o **LAB 01** antes (tabelas Delta no mesmo `catalog_name` / `schema_name`).
2. Confira os widgets.
3. Cluster ou serverless → **Run All**.
4. Use o Assistant (`Ctrl + I`) só com as colunas dos CSVs.
5. Não pergunte sobre faturas, CNPJ ou bairros — **não existem** nestes CSVs.

In [ ]:
def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "crm")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")
print(f"{catalog_name}.{schema_name}")

In [ ]:
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}"))

### Passo 1 — Perguntas abertas (Databricks Assistant)

1. Copie uma pergunta.
2. Clique no Assistant ao lado da célula SQL ou use `Ctrl + I`.
3. Peça SQL que use **somente** as colunas das tabelas `accounts`, `products`, `sales_teams`, `sales_pipeline`.
4. Execute o código sugerido.

`metadata` é dicionário. Pipe aberto = `Prospecting` + `Engaging`. `close_value` em aberto é null, não zero.

## Perguntas (colunas reais do CSV)

- Quantas oportunidades por `deal_stage`? Quantas abertas (Prospecting + Engaging)?
- `product` do pipeline que **não** existe em `products` (`GTXPro` vs `GTX Pro`).
- `account` nulo por stage — dá para pontuar deal aberto com inner join?
- Agentes em `sales_teams` sem linha em `sales_pipeline`.
- `close_value`: Lost é 0? Aberto é null? Won é > 0?
- Win rate (Won / (Won+Lost)) por `sector`, `regional_office`, `manager` e produto **depois** de mapear `GTXPro` → `GTX Pro`.
- Deals abertos com conta: `revenue` e `employees` da conta vs deals sem conta.
- Ordenar só por `close_value` serve para o pipe aberto? (dica: aberto = null).

In [ ]:
display(
    spark.sql(
        """
SELECT
  deal_stage,
  COUNT(*) AS opportunities,
  SUM(CASE WHEN account IS NULL THEN 1 ELSE 0 END) AS missing_account,
  SUM(CASE WHEN close_value IS NULL THEN 1 ELSE 0 END) AS missing_close_value,
  ROUND(AVG(close_value), 2) AS avg_close_value
FROM sales_pipeline
GROUP BY deal_stage
ORDER BY opportunities DESC
"""
    )
)

In [ ]:
display(
    spark.sql(
        """
SELECT
  p.product AS catalog_product,
  sp.product AS pipeline_product,
  COUNT(*) AS opportunities
FROM sales_pipeline sp
LEFT JOIN products p ON sp.product = p.product
GROUP BY p.product, sp.product
ORDER BY opportunities DESC
"""
    )
)

In [ ]:
display(
    spark.sql(
        """
SELECT t.sales_agent, t.manager, t.regional_office
FROM sales_teams t
LEFT JOIN (
  SELECT DISTINCT sales_agent FROM sales_pipeline
) p ON t.sales_agent = p.sales_agent
WHERE p.sales_agent IS NULL
ORDER BY t.sales_agent
"""
    )
)

### Bônus — prompt para o Assistant

> Estude `accounts`, `products`, `sales_teams` e `sales_pipeline`. Cruze por `account`, `product` e `sales_agent`. Mostre o órfão `GTXPro`. Não inner-join conta em deals abertos. Explique por que ordenar por `close_value` não prioriza o pipe aberto. Sugira 3 sinais (além do valor) para um vendedor na segunda-feira — com `opportunity_id` de exemplo. Não invente colunas.

### Genie

Sala publicada: [lab-003-crm-genie](https://dbc-0dea39b3-be1e.cloud.databricks.com/genie/rooms/01f1b6165c871b51a8ec3e632bffa2ac?o=7474644277755368) (`01f1b6165c871b51a8ec3e632bffa2ac`).

O chatbot local (`e2e-chatbot-app-next`) abre esta sala em `/room/crm`.

GTXPro != GTX Pro. close_value aberto e null.
